#  Electricity Demand Forecasting using LSTM

**Goal:** Predict future electricity demand using historical usage, temperature, and time features.

**Pipeline:**
```
Raw Data → Preprocessing → Normalization → Sequences → LSTM Model → Predictions → Visualization
```

---

## Step 1: Install & Import Libraries

In [ ]:
# Standard libraries — all pre-installed in Colab, no pip install needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn: for scaling (normalizing) the data
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TensorFlow/Keras: for building the LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Fix random seed so results are reproducible every run
np.random.seed(42)
tf.random.set_seed(42)

print(" Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

---
## Step 2: Generate Synthetic Dataset



In [ ]:
# ---------------------------------------------------------------
# Generate 2 years of hourly data (2022-01-01 to 2023-12-31)
# ---------------------------------------------------------------
hours = 24 * 365 * 2          # total hours in 2 years
date_range = pd.date_range(start='2022-01-01', periods=hours, freq='H')

# Time-based features
hour_of_day  = date_range.hour                  # 0–23
day_of_week  = date_range.dayofweek             # 0=Monday, 6=Sunday
day_of_year  = date_range.dayofyear             # 1–365

# --- Simulate Temperature (°C) ---
# Seasonal base: peaks ~35°C in summer (July), lowest ~5°C in winter
seasonal_temp = 20 + 15 * np.sin(2 * np.pi * (day_of_year - 172) / 365)
# Add daily variation: cooler at night, warmer in afternoon
daily_temp    = 5  * np.sin(2 * np.pi * (hour_of_day - 6) / 24)
# Add random noise
noise_temp    = np.random.normal(0, 2, hours)
temperature   = seasonal_temp + daily_temp + noise_temp

# --- Simulate Electricity Demand (MW) ---
# Base demand
base_demand = 500

# Daily pattern: morning ramp-up (8am), afternoon dip, evening peak (7pm)
daily_pattern = (80  * np.sin(2 * np.pi * (hour_of_day - 8)  / 24) +
                 40  * np.sin(4 * np.pi * (hour_of_day - 10) / 24))

# Weekly pattern: ~10% less demand on weekends (days 5 & 6)
weekend_effect = np.where(day_of_week >= 5, -50, 0)

# Seasonal pattern: high demand in summer and winter (air conditioning + heating)
seasonal_demand = 60 * np.abs(np.sin(2 * np.pi * day_of_year / 365))

# Temperature effect: demand rises when very hot (>30°C) or very cold (<5°C)
temp_effect = 3 * np.maximum(0, temperature - 30) + 4 * np.maximum(0, 5 - temperature)

# Random noise
noise_demand = np.random.normal(0, 15, hours)

# Combine all components
electricity_demand = (base_demand + daily_pattern + weekend_effect +
                      seasonal_demand + temp_effect + noise_demand)

# Clip to realistic range: no negative demand
electricity_demand = np.clip(electricity_demand, 200, 900)

# ---------------------------------------------------------------
# Build the DataFrame
# ---------------------------------------------------------------
df = pd.DataFrame({
    'datetime'    : date_range,
    'temperature' : np.round(temperature, 2),
    'hour'        : hour_of_day,
    'day_of_week' : day_of_week,
    'demand_mw'   : np.round(electricity_demand, 2)
})
df.set_index('datetime', inplace=True)

print(f" Dataset created: {df.shape[0]:,} hourly records")
print(f"   Date range : {df.index[0]} → {df.index[-1]}")
print(f"   Demand range: {df['demand_mw'].min():.0f} MW – {df['demand_mw'].max():.0f} MW")
df.head(10)

---
## Step 3: Exploratory Data Analysis (EDA)



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('Electricity Demand — Exploratory Analysis', fontsize=15, fontweight='bold')

# --- Plot 1: Full 2-year demand (resampled to daily mean for readability) ---
df['demand_mw'].resample('D').mean().plot(ax=axes[0], color='steelblue', linewidth=0.8)
axes[0].set_title('Daily Average Demand (2 Years)')
axes[0].set_ylabel('Demand (MW)')
axes[0].grid(alpha=0.3)

# --- Plot 2: One week of hourly data — shows daily peaks/troughs ---
df['demand_mw'].iloc[:24*7].plot(ax=axes[1], color='darkorange', linewidth=1.2)
axes[1].set_title('Hourly Demand — First Week')
axes[1].set_ylabel('Demand (MW)')
axes[1].grid(alpha=0.3)

# --- Plot 3: Temperature vs Demand scatter ---
axes[2].scatter(df['temperature'], df['demand_mw'], alpha=0.05, s=5, color='teal')
axes[2].set_title('Temperature vs Demand (shows V-shape: hot & cold = high demand)')
axes[2].set_xlabel('Temperature (°C)')
axes[2].set_ylabel('Demand (MW)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 4: Normalize the Data



In [ ]:
# Select only the features the model will use
features = ['temperature', 'hour', 'day_of_week', 'demand_mw']
data = df[features].values   # shape: (hours, 4) as numpy array

# Scale all columns to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# We need a separate scaler just for demand (to inverse-transform predictions later)
demand_scaler = MinMaxScaler(feature_range=(0, 1))
demand_scaler.fit(data[:, -1].reshape(-1, 1))   # fit only on demand column

print(f" Data normalized. Shape: {data_scaled.shape}")
print(f"   Each value is now between {data_scaled.min():.2f} and {data_scaled.max():.2f}")

---
## Step 5: Create Sequences (Sliding Window)



In [ ]:
def create_sequences(data, window_size):
    """
    Creates (X, y) pairs using a sliding window.
    X: past `window_size` time steps (all features)
    y: next time step's demand (last column)
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size : i, :])   # all features, past window
        y.append(data[i, -1])                    # only demand, next step
    return np.array(X), np.array(y)

# Use 24 hours (1 day) as the look-back window
WINDOW_SIZE = 24
X, y = create_sequences(data_scaled, WINDOW_SIZE)

# ---------------------------------------------------------------
# Train/Test Split — use last 30 days (~720 hours) as test set
# ---------------------------------------------------------------
TEST_SIZE   = 24 * 30     # 720 hours = 30 days
TRAIN_SIZE  = len(X) - TEST_SIZE

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE:]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE:]

print(f"✅ Sequences created with window size = {WINDOW_SIZE} hours")
print(f"   X shape : {X.shape}  → (samples, time_steps, features)")
print(f"   y shape : {y.shape}  → (samples,)")
print(f"   Training samples : {X_train.shape[0]:,}")
print(f"   Testing  samples : {X_test.shape[0]:,}  ({TEST_SIZE/24:.0f} days)")

---
## Step 6: Build the LSTM Model

> **Architecture (keep it simple):**
> - **Layer 1:** LSTM with 64 units — learns patterns from the 24-hour input sequence
> - **Layer 2:** Dense(32) — a small fully-connected layer to refine the output
> - **Layer 3:** Dense(1) — outputs a single value: the predicted demand
>
> **Dropout(0.2):** randomly drops 20% of neurons during training → prevents overfitting

In [ ]:
# Input shape: (time_steps=24, features=4)
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])

model = Sequential([
    # LSTM layer: processes the 24-step sequence, returns a single vector
    LSTM(64, input_shape=INPUT_SHAPE, return_sequences=False),

    # Dropout: ignore 20% of neurons randomly to reduce overfitting
    Dropout(0.2),

    # Dense hidden layer: further transforms the LSTM output
    Dense(32, activation='relu'),

    # Output layer: predict a single value (next hour's scaled demand)
    Dense(1)
])

# Compile: Adam optimizer + MSE loss (standard for regression tasks)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Print model summary
model.summary()

---
## Step 7: Train the Model

> **Key parameters:**
> - `epochs=20`: the model sees the full training data 20 times
> - `batch_size=64`: updates weights after every 64 samples
> - `validation_split=0.1`: uses 10% of training data to monitor overfitting
> - `EarlyStopping`: automatically stops training if validation loss stops improving (saves time)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training early if val_loss doesn't improve for 5 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,   # revert to the best model found
    verbose=1
)

print("🚀 Training the LSTM model...")
history = model.fit(
    X_train, y_train,
    epochs          = 20,
    batch_size      = 64,
    validation_split= 0.1,
    callbacks       = [early_stop],
    verbose         = 1
)
print("\n✅ Training complete!")

---
## Step 8: Plot Training History

> If both `loss` and `val_loss` go down together, the model is learning well and not overfitting.

In [ ]:
plt.figure(figsize=(10, 4))

# Training loss vs Validation loss per epoch
plt.plot(history.history['loss'],     label='Training Loss',   color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='darkorange', linestyle='--')

plt.title('Model Loss During Training')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 9: Evaluate the Model on Test Data

> **Metrics explained:**
> - **MAE (Mean Absolute Error):** average error in MW — lower is better
> - **RMSE (Root Mean Squared Error):** penalizes large errors more — lower is better
> - **MAPE (Mean Absolute Percentage Error):** error as % — easier to interpret

In [ ]:
# Generate predictions on test set (still in scaled form)
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse-transform: convert predictions back to original MW scale
y_pred = demand_scaler.inverse_transform(y_pred_scaled).flatten()
y_true = demand_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Calculate metrics
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("📊 Model Performance on Test Set (Last 30 Days):")
print(f"   MAE  (Mean Absolute Error)      : {mae:.2f} MW")
print(f"   RMSE (Root Mean Squared Error)  : {rmse:.2f} MW")
print(f"   MAPE (Mean Abs Percentage Error): {mape:.2f}%")
print(f"\n   Average actual demand           : {y_true.mean():.2f} MW")
print(f"   → The model is off by ~{mape:.1f}% on average")

---
## Step 10: Visualize — Actual vs Predicted

> Plot the model's predictions against real values. A good model's line will closely follow the actual values.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Actual vs Predicted Electricity Demand', fontsize=15, fontweight='bold')

# ---------------------------------------------------------------
# Plot 1: Full 30-day test period
# ---------------------------------------------------------------
axes[0].plot(y_true, label='Actual Demand',    color='steelblue',  linewidth=1.0, alpha=0.9)
axes[0].plot(y_pred, label='Predicted Demand', color='darkorange', linewidth=1.0, alpha=0.85, linestyle='--')
axes[0].set_title(f'Full Test Period (30 days) — MAPE: {mape:.2f}%')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ---------------------------------------------------------------
# Plot 2: Zoom into first 7 days — easier to see pattern matching
# ---------------------------------------------------------------
zoom = 24 * 7    # first 168 hours
axes[1].plot(y_true[:zoom], label='Actual Demand',    color='steelblue',  linewidth=1.2)
axes[1].plot(y_pred[:zoom], label='Predicted Demand', color='darkorange', linewidth=1.2, linestyle='--')
axes[1].set_title('Zoomed In — First 7 Days of Test Period')
axes[1].set_xlabel('Hours')
axes[1].set_ylabel('Demand (MW)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 11: Predict Next 24 Hours (Next-Day Forecast)

> **How it works:** We feed the last 24 hours of real data into the model, get a prediction for hour 25,
> then roll the window forward to predict hour 26, and so on (**multi-step forecasting**).

In [ ]:
def forecast_next_n_hours(model, last_window, n_hours, scaler, demand_scaler, n_features):
    """
    Predict the next n_hours demand values using rolling/recursive forecasting.
    last_window: the most recent 'window_size' rows of scaled data
    """
    predictions = []
    current_input = last_window.copy()   # shape: (window_size, n_features)

    for _ in range(n_hours):
        # Reshape for LSTM: (1, window_size, n_features)
        x_input = current_input.reshape(1, current_input.shape[0], current_input.shape[1])

        # Predict next scaled demand value
        next_pred_scaled = model.predict(x_input, verbose=0)[0, 0]

        # Inverse transform to get MW value
        next_pred_mw = demand_scaler.inverse_transform([[next_pred_scaled]])[0, 0]
        predictions.append(next_pred_mw)

        # Roll the window: drop oldest row, append new row with predicted demand
        # For simplicity, we copy the last row and update the demand column
        new_row          = current_input[-1].copy()
        new_row[-1]      = next_pred_scaled      # update demand with prediction
        current_input    = np.vstack([current_input[1:], new_row])  # slide window

    return np.array(predictions)


# Use the very last 24 hours of our data as the starting window
last_window = data_scaled[-WINDOW_SIZE:]

# Forecast next 24 hours (1 day)
forecast_24h = forecast_next_n_hours(
    model, last_window, n_hours=24,
    scaler=scaler, demand_scaler=demand_scaler,
    n_features=data_scaled.shape[1]
)

# Create future timestamps (starting from last data point)
last_time   = df.index[-1]
future_times = pd.date_range(start=last_time + pd.Timedelta(hours=1), periods=24, freq='H')

print("✅ Next 24-Hour Demand Forecast:")
forecast_df = pd.DataFrame({'Datetime': future_times, 'Forecasted_Demand_MW': np.round(forecast_24h, 1)})
forecast_df.set_index('Datetime', inplace=True)
print(forecast_df.to_string())

---
## Step 12: Plot Next-Day & Next-Week Forecast

In [ ]:
# Forecast next 7 days (168 hours)
forecast_168h = forecast_next_n_hours(
    model, last_window, n_hours=168,
    scaler=scaler, demand_scaler=demand_scaler,
    n_features=data_scaled.shape[1]
)
future_times_week = pd.date_range(start=last_time + pd.Timedelta(hours=1), periods=168, freq='H')

# Recent actual data (last 7 days) for context
recent_actual = df['demand_mw'].iloc[-168:]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Electricity Demand Forecast', fontsize=15, fontweight='bold')

# ---------------------------------------------------------------
# Plot 1: Next 24 hours forecast
# ---------------------------------------------------------------
# Show 48h actual context + 24h forecast
context_48h = df['demand_mw'].iloc[-48:]
axes[0].plot(context_48h.index, context_48h.values,
             label='Recent Actual', color='steelblue', linewidth=1.5)
axes[0].plot(future_times[:24], forecast_24h,
             label='24h Forecast', color='crimson', linewidth=2, linestyle='--', marker='o', markersize=4)
axes[0].axvline(x=last_time, color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[0].set_title('Next 24-Hour Demand Forecast')
axes[0].set_ylabel('Demand (MW)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ---------------------------------------------------------------
# Plot 2: Next 7 days forecast
# ---------------------------------------------------------------
axes[1].plot(recent_actual.index, recent_actual.values,
             label='Recent Actual (last 7 days)', color='steelblue', linewidth=1.2)
axes[1].plot(future_times_week, forecast_168h,
             label='7-Day Forecast', color='crimson', linewidth=1.5, linestyle='--')
axes[1].axvline(x=last_time, color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[1].set_title('Next 7-Day Demand Forecast')
axes[1].set_ylabel('Demand (MW)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("\n✅ Forecast plots generated!")

---
## Step 13: Scatter Plot — Predicted vs Actual

> A perfect model would have all dots on the red diagonal line (predicted = actual).

In [ ]:
plt.figure(figsize=(7, 7))

# Scatter: each dot is one test hour
plt.scatter(y_true, y_pred, alpha=0.3, s=8, color='steelblue', label='Predicted vs Actual')

# Perfect prediction line
min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.title(f'Predicted vs Actual Demand\nMAE={mae:.1f} MW | MAPE={mape:.1f}%')
plt.xlabel('Actual Demand (MW)')
plt.ylabel('Predicted Demand (MW)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## ✅ Summary — What We Built

| Step | What We Did |
|------|-------------|
| 1 | Imported TensorFlow, NumPy, Pandas, Matplotlib |
| 2 | Generated synthetic hourly electricity + temperature data |
| 3 | Visualized demand patterns (daily, weekly, seasonal) |
| 4 | Normalized all features to [0,1] using MinMaxScaler |
| 5 | Created 24-hour sliding window sequences for LSTM |
| 6 | Built simple LSTM → Dropout → Dense → Dense model |
| 7 | Trained with EarlyStopping on 80/20 train-test split |
| 8 | Plotted training vs validation loss |
| 9 | Evaluated: MAE, RMSE, MAPE on test data |
| 10 | Visualized actual vs predicted demand |
| 11-12 | Forecasted next 24 hours and next 7 days |
| 13 | Scatter plot of predictions vs actuals |

---

## 🎤 Interview Q&A Cheatsheet

**Q: Why LSTM for time-series?**
> LSTM has a "memory" (cell state) that lets it remember patterns over long sequences — unlike plain neural nets that forget earlier inputs.

**Q: Why normalize data?**
> Neural networks are sensitive to input scale. MinMaxScaler brings all features to [0,1], making training faster and more stable.

**Q: What is a sliding window?**
> We use the past N time steps as input to predict the next step. Sliding it forward creates many training examples from one time series.

**Q: What does Dropout do?**
> It randomly ignores some neurons during training, forcing the network to learn more robust patterns and reducing overfitting.

**Q: How would you improve this model?**
> Add more features (humidity, holidays), use a larger window, add a second LSTM layer, or try Transformer-based models.

---
*Built with TensorFlow/Keras · Synthetic Data · Beginner-Friendly LSTM Pipeline*